# E2E-DRO replication — parallel retraining on a Colab TPU runtime

**为什么用 TPU runtime / Why the TPU runtime**: we do **not** use the TPU itself.
The TPU runtime is selected purely because it ships with many vCPUs, and this
workload is CPU-bound — cvxpylayers/diffcp solves and differentiates thousands of
small conic programs. Do not install or import `torch_xla`; plain CPU torch is
what we want.

**Runtime → Change runtime type → TPU**, then run the cells in order.

**Why this works at all** (the structural finding): `net_roll_test` reloads the
same saved init state and re-fits the prediction layer to OLS at the top of every
roll window, so the 4 windows share no state. They are independent jobs. Total
work is ~146,700 forward+backward passes per net (~16 h serial, which is why the
earlier single-threaded attempt could never fit in a Colab session), but the
windows tile as 114+113+114+113 = 454 OOS weeks and run concurrently — so
wall-clock is the **slowest single roll** (roll 3, ~45,200 steps), not the sum.

**Resilience**: every job checkpoints after each epoch. If the session drops,
re-run the launch cell — finished jobs are skipped and partial ones resume from
their last epoch. Mount Drive (cell 3) to keep checkpoints across sessions.

In [ ]:
# 1) Runtime check + dependencies
import os, multiprocessing
print("vCPUs:", multiprocessing.cpu_count())
!nproc; free -g | head -2
# ecos: newer cvxpy no longer bundles it but the layers still request it.
%pip -q install cvxpylayers "cvxpy>=1.4" ecos pandas_datareader alpha_vantage statsmodels psutil
import torch, cvxpy
print("torch", torch.__version__, "| cvxpy", cvxpy.__version__)
assert not torch.cuda.is_available() or True  # GPU irrelevant; CPU path is used

In [ ]:
# 2) Clone the upstream repo (ships the real dataset in cache/*.pkl)
import os, sys, subprocess
REPO = "/content/E2E-DRO"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Iyengar-Lab/E2E-DRO.git {REPO}
os.makedirs(REPO + "/new_cache/exp", exist_ok=True)
assert os.path.isdir(REPO + "/e2edro") and os.path.isdir(REPO + "/cache")
print("repo OK:", REPO)
!ls {REPO}/cache/

In [ ]:
# 3) OPTIONAL but recommended — persist chunks/checkpoints on Drive so a
# disconnect does not lose hours of training. Skip if you prefer ephemeral runs.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHUNKS = '/content/drive/MyDrive/polab_rolls'
else:
    CHUNKS = '/content/polab_rolls'
LOGS = '/content/polab_logs'
os.makedirs(CHUNKS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
print("chunks ->", CHUNKS)
print("logs   ->", LOGS)

In [ ]:
# 4) Write the polab.rolls module (split arithmetic; verified offline
#    against their 454 OOS weeks)
import os
os.makedirs('/content/polab', exist_ok=True)
open('/content/polab/__init__.py','w').write('')
_src = r'''"""Roll-window bookkeeping for divide-and-conquer retraining of E2E-DRO.

`e2edro.e2e_net.net_roll_test` loops over `n_roll` windows. Each iteration
reloads the SAME saved init state and re-fits the prediction layer to OLS on
that window's training data, so **no state carries between windows** — they are
independent jobs that can run in separate processes, in parallel, and be
resumed individually.

This module reproduces their split arithmetic in pure pandas/numpy (no torch,
no cvxpylayers) so the plan can be verified offline and each worker knows where
its slice belongs in the assembled backtest.

Reference (their code, verbatim logic):
    win_size = init_split[1] / n_roll
    split[0] = init_split[0] + win_size * i
    split[1] = win_size            if i < n_roll-1  else  1 - split[0]
    numel    = round(n_total * cumsum(split))
    train    = data[:numel[0]]
    test     = data[numel[0]-n_obs : numel[1]]
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np


@dataclass(frozen=True)
class RollWindow:
    index: int              # roll number, 0-based
    split: tuple            # (train_frac, test_frac) passed to split_update
    train_end: int          # numel[0]: rows of training data
    test_end: int           # numel[1]
    n_train_windows: int    # sliding windows the trainer iterates per epoch
    n_test_windows: int     # OOS weeks this roll contributes
    offset: int             # where this roll's returns start in the full backtest


def _numel(n_total: int, split) -> list[int]:
    return [round(v) for v in n_total * np.cumsum(split)]


def plan(n_total: int = 1134, init_split=(0.6, 0.4), n_roll: int = 4,
         n_obs: int = 104, perf_period: int = 13) -> list[RollWindow]:
    """Enumerate the roll windows exactly as their loop would."""
    win_size = init_split[1] / n_roll
    out, offset = [], 0
    for i in range(n_roll):
        s0 = init_split[0] + win_size * i
        s1 = win_size if i < n_roll - 1 else 1 - s0
        numel = _numel(n_total, (s0, s1))
        n_test = (numel[1] - (numel[0] - n_obs)) - n_obs
        # SlidingWindow(train, n_obs, perf_period) yields this many batches
        n_train = numel[0] - n_obs - perf_period
        out.append(RollWindow(i, (s0, s1), numel[0], numel[1],
                              n_train, n_test, offset))
        offset += n_test
    return out


def total_test_windows(n_total: int = 1134, init_split=(0.6, 0.4),
                       n_obs: int = 104) -> int:
    """Length of the assembled backtest (their `pc.backtest(...)` sizing)."""
    numel = _numel(n_total, init_split)
    return (numel[1] - (numel[0] - n_obs)) - n_obs


def summary(rolls: list[RollWindow], epochs: int = 50) -> str:
    lines = [f"{'roll':>4} {'train_win':>9} {'test_win':>8} {'offset':>6} "
             f"{'steps@%d ep' % epochs:>12}"]
    for r in rolls:
        lines.append(f"{r.index:>4} {r.n_train_windows:>9} {r.n_test_windows:>8} "
                     f"{r.offset:>6} {r.n_train_windows * epochs:>12,}")
    tot_steps = sum(r.n_train_windows for r in rolls) * epochs
    lines.append(f"{'ALL':>4} {'':>9} {sum(r.n_test_windows for r in rolls):>8} "
                 f"{'':>6} {tot_steps:>12,}")
    return "\n".join(lines)
'''
open('/content/polab/rolls.py','w').write(_src)
import sys; sys.path.insert(0, '/content')
from polab import rolls as R
print(R.summary(R.plan()))

In [ ]:
# 5) Write the worker script (one (net, roll) job, resumable)
os.makedirs('/content/scripts', exist_ok=True)
_src = r'''"""Train ONE (net, roll-window) job of the E2E-DRO replication.

Their `net_roll_test` reloads the same init state and re-fits the prediction
layer to OLS at the top of every roll window, so the windows share no state and
can run as independent processes. This script runs exactly one of them and
writes a self-contained chunk; `combine_rolls.py` stitches the chunks back into
a full backtest.

Faithfulness: the training loop below is a line-by-line reimplementation of
`e2edro.e2e_net.net_train` (full-batch gradient accumulation, one Adam step per
epoch, gamma/delta clamped at 1e-4), with per-epoch checkpointing added so a
crash or a killed session loses at most one epoch.

Requires the dedicated env (torch + cvxpy + cvxpylayers + ecos), NOT anaconda base:
    python3 -m venv ~/.venvs/polab && source ~/.venvs/polab/bin/activate
    pip install torch cvxpy cvxpylayers ecos pandas scipy pandas_datareader \\
                alpha_vantage statsmodels psutil

Usage:
    python scripts/train_roll.py --net nom --roll 0
    # all 8 jobs in parallel, 1 torch thread each (the conic solves dominate):
    for n in nom dr; do for r in 0 1 2 3; do
        OMP_NUM_THREADS=1 python scripts/train_roll.py --net $n --roll $r &
    done; done; wait
"""

from __future__ import annotations

import argparse
import os
import pickle
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
# POLAB_VENDOR lets a foreign layout (e.g. Colab, where the upstream repo sits at
# /content/E2E-DRO) point at the clone without mirroring our directory tree.
VENDOR = Path(os.environ.get("POLAB_VENDOR", str(ROOT / "vendor" / "E2E-DRO")))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(VENDOR))

from polab import rolls as R  # noqa: E402

# Net configurations, verbatim from their main.py + the CV winners we recovered
# from their cached objects (cv grid skipped: we pass the winning lr/epochs).
NET_CONFIGS = {
    "base": dict(opt_layer="base_mod", train_pred=True, train_gamma=False,
                 train_delta=False, lr=0.005, epochs=30, pkl="base_net"),
    "nom": dict(opt_layer="nominal", train_pred=True, train_gamma=True,
                train_delta=False, lr=0.02, epochs=50, pkl="nom_net"),
    "dr": dict(opt_layer="hellinger", train_pred=True, train_gamma=True,
               train_delta=True, lr=0.0125, epochs=50, pkl="dr_net"),
    "dr_theta": dict(opt_layer="hellinger", train_pred=True, train_gamma=False,
                     train_delta=False, lr=0.0125, epochs=40,
                     pkl="dr_net_learn_theta"),
}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--net", choices=sorted(NET_CONFIGS), required=True)
    ap.add_argument("--roll", type=int, required=True)
    ap.add_argument("--n-roll", type=int, default=4)
    ap.add_argument("--epochs", type=int, default=None, help="override CV winner")
    ap.add_argument("--lr", type=float, default=None, help="override CV winner")
    ap.add_argument("--threads", type=int, default=1,
                    help="torch threads; keep at 1 when running jobs in parallel")
    ap.add_argument("--out", default=str(ROOT / "results" / "rolls"))
    args = ap.parse_args()

    cfg = NET_CONFIGS[args.net]
    lr = args.lr if args.lr is not None else cfg["lr"]
    epochs = args.epochs if args.epochs is not None else cfg["epochs"]
    tag = f"{cfg['pkl']}_roll{args.roll}"

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    chunk_path = out_dir / f"{tag}.pkl"
    ckpt_path = out_dir / f"{tag}.ckpt"
    if chunk_path.exists():
        print(f"{tag}: already complete ({chunk_path}) — nothing to do")
        return

    os.chdir(VENDOR)  # their relative paths (./cache/...) resolve from here
    import torch
    torch.set_num_threads(max(1, args.threads))

    # PyTorch >=2.6 defaults torch.load to weights_only=True; the only things we
    # load are state dicts written by this process.
    _orig_load = torch.load
    torch.load = lambda *a, **k: _orig_load(*a, **{**k, "weights_only": False})

    from torch.utils.data import DataLoader
    from torch.autograd import Variable
    from e2edro import e2edro as e2e
    from e2edro import DataLoad as dl
    from e2edro import PortfolioClasses as pc

    # ---- data (their exact call; ships in the vendored cache) ----------------
    init_split = [0.6, 0.4]
    n_obs, perf_period = 104, 13
    X, Y = dl.AV("2000-01-01", "2021-09-30", init_split, freq="weekly",
                 n_obs=n_obs, n_y=20, use_cache=True, save_results=False,
                 AV_key=None)
    n_x, n_y = X.data.shape[1], Y.data.shape[1]

    plan = R.plan(n_total=X.data.shape[0], init_split=tuple(init_split),
                  n_roll=args.n_roll, n_obs=n_obs, perf_period=perf_period)
    win = plan[args.roll]
    print(f"{tag}: split={win.split} train_win={win.n_train_windows} "
          f"test_win={win.n_test_windows} offset={win.offset} "
          f"lr={lr} epochs={epochs}")

    cache_path = str(VENDOR / "new_cache" / "exp") + "/"
    Path(cache_path).mkdir(parents=True, exist_ok=True)
    net = e2e.e2e_net(n_x, n_y, n_obs, prisk="p_var",
                      train_pred=cfg["train_pred"], train_gamma=cfg["train_gamma"],
                      train_delta=cfg["train_delta"], set_seed=1000,
                      opt_layer=cfg["opt_layer"], perf_loss="sharpe_loss",
                      cache_path=cache_path, perf_period=perf_period,
                      pred_loss_factor=0.5).double()

    # ---- roll-window setup (their loop body, verbatim order) ----------------
    X.split_update(list(win.split)), Y.split_update(list(win.split))
    train_set = DataLoader(pc.SlidingWindow(X.train(), Y.train(), n_obs, perf_period))
    test_set = DataLoader(pc.SlidingWindow(X.test(), Y.test(), n_obs, 0))
    assert len(train_set) == win.n_train_windows, "train window count mismatch"
    assert len(test_set) == win.n_test_windows, "test window count mismatch"

    net.load_state_dict(torch.load(net.init_state_path))

    # prediction layer initialized to the OLS solution on this window
    X_train, Y_train = X.train(), Y.train()
    X_train.insert(0, "ones", 1.0)
    X_train = Variable(torch.tensor(X_train.values, dtype=torch.double))
    Y_train = Variable(torch.tensor(Y_train.values, dtype=torch.double))
    Theta = (torch.inverse(X_train.T @ X_train) @ (X_train.T @ Y_train)).T
    del X_train, Y_train
    with torch.no_grad():
        net.pred_layer.bias.copy_(Theta[:, 0])
        net.pred_layer.weight.copy_(Theta[:, 1:])

    # ---- training (reimplements net_train + checkpointing) ------------------
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    n_train = len(train_set)
    start_epoch = 0
    if ckpt_path.exists():
        ck = torch.load(ckpt_path)
        net.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optim"])
        start_epoch = ck["epoch"] + 1
        print(f"{tag}: resuming from epoch {start_epoch}")

    t_start = time.time()
    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        train_loss = 0.0
        optimizer.zero_grad()
        for x, y, y_perf in train_set:
            z_star, y_hat = net(x.squeeze(), y.squeeze())
            if net.pred_loss is None:
                loss = (1 / n_train) * net.perf_loss(z_star, y_perf.squeeze())
            else:
                loss = (1 / n_train) * (
                    net.perf_loss(z_star, y_perf.squeeze())
                    + (net.pred_loss_factor / net.n_y)
                    * net.pred_loss(y_hat, y_perf.squeeze()[0]))
            loss.backward()
            train_loss += loss.item()
        optimizer.step()
        for name, param in net.named_parameters():
            if name in ("gamma", "delta"):
                param.data.clamp_(0.0001)

        torch.save({"model": net.state_dict(), "optim": optimizer.state_dict(),
                    "epoch": epoch}, ckpt_path)
        done, left = epoch - start_epoch + 1, epochs - epoch - 1
        eta = (time.time() - t_start) / done * left / 60
        print(f"{tag}: epoch {epoch + 1}/{epochs} loss={train_loss:.6f} "
              f"({time.time() - t0:.1f}s, ETA {eta:.0f} min)", flush=True)

    # ---- out-of-sample evaluation of this window ---------------------------
    weights = np.zeros((win.n_test_windows, n_y))
    rets = np.zeros(win.n_test_windows)
    with torch.no_grad():
        for t, (x, y, y_perf) in enumerate(test_set):
            z_star, _ = net(x.squeeze(), y.squeeze())
            weights[t] = z_star.squeeze()
            rets[t] = y_perf.squeeze() @ weights[t]

    chunk = {
        "net": cfg["pkl"], "roll": args.roll, "offset": win.offset,
        "dates": Y.test().index[n_obs:], "weights": weights, "rets": rets,
        "gamma": net.gamma.item() if cfg["train_gamma"] or cfg["opt_layer"] != "base_mod"
                 else None,
        "delta": net.delta.item() if hasattr(net, "delta") else None,
        "lr": lr, "epochs": epochs, "split": win.split,
        "train_seconds": time.time() - t_start,
    }
    with open(chunk_path, "wb") as f:
        pickle.dump(chunk, f)
    ckpt_path.unlink(missing_ok=True)
    print(f"{tag}: DONE -> {chunk_path} "
          f"({chunk['train_seconds'] / 3600:.2f} h, mean ret {rets.mean():.5f})")


if __name__ == "__main__":
    main()
'''
open('/content/scripts/train_roll.py','w').write(_src)
print('worker written:', len(_src), 'bytes')

In [ ]:
# 6) DIAGNOSTIC — 1 forward+backward, ~1 min. Confirms the training path
# works and gives a real per-step time so the ETA below is grounded.
import os, sys, time, traceback, torch
os.chdir(REPO); sys.path.insert(0, REPO)
torch.set_num_threads(1)
_orig = torch.load
torch.load = lambda *a, **k: _orig(*a, **{**k, "weights_only": False})

from torch.utils.data import DataLoader
from e2edro import e2edro as e2e, DataLoad as dl, PortfolioClasses as pc

X, Y = dl.AV("2000-01-01", "2021-09-30", [0.6, 0.4], freq="weekly",
             n_obs=104, n_y=20, use_cache=True, save_results=False, AV_key=None)
n_x, n_y = X.data.shape[1], Y.data.shape[1]
print("data:", X.data.shape, Y.data.shape)

diag = e2e.e2e_net(n_x, n_y, 104, prisk="p_var", train_pred=True, train_gamma=True,
                   train_delta=False, set_seed=1000, opt_layer="nominal",
                   perf_loss="sharpe_loss", cache_path=REPO + "/new_cache/exp/",
                   perf_period=13, pred_loss_factor=0.5).double()
diag.load_state_dict(torch.load(diag.init_state_path))
print("init state reload: OK")

X.split_update([0.6, 0.1]); Y.split_update([0.6, 0.1])
ts = DataLoader(pc.SlidingWindow(X.train(), Y.train(), 104, 13))
x, y, y_perf = next(iter(ts))
try:
    t0 = time.time(); z, yh = diag(x.squeeze(), y.squeeze()); tf = time.time() - t0
    loss = diag.perf_loss(z, y_perf.squeeze())
    t0 = time.time(); loss.backward(); tb = time.time() - t0
    print(f"forward {tf:.2f}s + backward {tb:.2f}s = {tf+tb:.2f}s/step")
    print(f"slowest roll (roll 3, 45,200 steps) ETA: "
          f"{45200*(tf+tb)/3600:.1f} h  <-- this is the wall-clock to expect")
except Exception:
    traceback.print_exc()
X.split_update([0.6, 0.4]); Y.split_update([0.6, 0.4])

In [ ]:
# 7) LAUNCH — all (net, roll) jobs in parallel, one process each.
# Each job is pinned to 1 thread: the conic solves are single-threaded, so N
# processes on N cores scale nearly linearly. Re-run this cell after a
# disconnect — completed jobs are skipped, partial ones resume.
import subprocess, sys, os

NETS = ["nom", "dr", "dr_theta"]     # the headline comparison + the cost-stress winner
ROLLS = [0, 1, 2, 3]

env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
       "OPENBLAS_NUM_THREADS": "1", "POLAB_VENDOR": REPO}
procs = {}
for net in NETS:
    for r in ROLLS:
        tag = f"{net}_roll{r}"
        # No skip logic needed: the worker exits immediately if its chunk
        # already exists, and resumes from the last epoch if a .ckpt is there.
        log = open(f"{LOGS}/{tag}.log", "a")
        procs[tag] = subprocess.Popen(
            [sys.executable, "/content/scripts/train_roll.py",
             "--net", net, "--roll", str(r), "--out", CHUNKS, "--threads", "1"],
            stdout=log, stderr=subprocess.STDOUT, env=env, cwd="/content")
print(f"launched {len(procs)} jobs on {os.cpu_count()} vCPUs "
      f"(already-complete ones will exit within seconds)")

In [ ]:
# 8) MONITOR — re-runnable. Keep this cell running to hold the session open.
import time, glob, os
while True:
    alive = {t: p for t, p in procs.items() if p.poll() is None}
    done = sorted(os.path.basename(f) for f in glob.glob(f"{CHUNKS}/*.pkl"))
    print(f"[{time.strftime('%H:%M:%S')}] running={len(alive)} chunks={len(done)}")
    for tag in sorted(procs):
        log = f"{LOGS}/{tag}.log"
        last = ""
        if os.path.exists(log):
            lines = [l.strip() for l in open(log).readlines() if l.strip()]
            last = lines[-1][-90:] if lines else ""
        state = "RUN " if procs[tag].poll() is None else f"EXIT{procs[tag].poll()}"
        print(f"  {state} {tag:22s} {last}")
    if not alive:
        print("all jobs finished"); break
    time.sleep(120)

In [ ]:
# 9) Package the chunks for download (combine + scoring happens LOCALLY,
# where the vendored cache and the full polab package live)
import shutil
shutil.make_archive('/content/polab_chunks', 'zip', CHUNKS)
!ls -la /content/polab_chunks.zip
from google.colab import files
files.download('/content/polab_chunks.zip')

## 带回本地 / Bring the results home

```bash
unzip -o ~/Downloads/polab_chunks.zip -d research-projects/portfolio-lab/results/rolls/
python scripts/combine_rolls.py
```

`combine_rolls.py` stitches the four windows per net (verifying the assembled
length is exactly 454), then reports retrained vs shipped-cache Sharpe.

**Success criterion**: the ranking `dr_net > nom_net > 1/N > po_net` is preserved
and the Sharpes land near the cached reference (dr 1.314, nom 1.178,
dr_learn_theta 1.414). Exact reproduction is *not* expected — torch/cvxpy/
cvxpylayers have all moved several versions since 2022, even with their seed
(1000). A large deviation is a **finding**, not a failure; record it either way
in `notes.md`.

**If the session drops**: reconnect, re-run cells 1–5 (fast), then cell 7. With
Drive mounted, finished jobs are skipped and partial ones resume from their last
checkpointed epoch.